# Empty-Prediction Ablation

Computes the char-F1 score that a model achieves by **always predicting an empty set** on ToxicSpans and LegalQAEval.

This is the baseline inflated by the empty-set convention: when both the gold annotation and the prediction are empty, `compute_character_f1` returns F1 = 1.0. The resulting score shows how much of the observed unconstrained-model F1 can be attributed to this metric artifact rather than genuine extraction.

In [1]:
import sys
sys.path.insert(0, '../src')

import ast
import statistics
from datasets import load_dataset
from utils.utils_functions import compute_character_f1, parse_position

## ToxicSpans

In [2]:
toxic = load_dataset("heegyu/toxic-spans", split="test")
print(f"ToxicSpans test examples: {len(toxic)}")

README.md: 0.00B [00:00, ?B/s]

train.csv: 0.00B [00:00, ?B/s]

test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/10006 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

ToxicSpans test examples: 1000


In [4]:
toxic

Dataset({
    features: ['probability', 'position', 'text', 'type', 'support', 'text_of_post', 'position_probability', 'toxic'],
    num_rows: 1000
})

In [5]:
toxic_f1_scores = []
n_empty_gold = 0

for ex in toxic:
    gold_chars = set(parse_position(ex["position"]) if isinstance(ex["position"], str) else ex["position"])
    if not gold_chars:
        n_empty_gold += 1
    _, _, f1 = compute_character_f1(gold_chars, set())
    toxic_f1_scores.append(f1)

toxic_mean_f1 = statistics.mean(toxic_f1_scores) * 100
print(f"Examples with empty gold annotation: {n_empty_gold} / {len(toxic)} ({n_empty_gold/len(toxic)*100:.1f}%)")
print(f"Empty-prediction char-F1 (ToxicSpans): {toxic_mean_f1:.2f}%")

Examples with empty gold annotation: 486 / 1000 (48.6%)
Empty-prediction char-F1 (ToxicSpans): 48.60%


## LegalQAEval

In [6]:
legal = load_dataset("isaacus/LegalQAEval", split="test")
print(f"LegalQAEval test examples: {len(legal)}")

README.md: 0.00B [00:00, ?B/s]

val.jsonl: 0.00B [00:00, ?B/s]

test.jsonl: 0.00B [00:00, ?B/s]

Generating val split:   0%|          | 0/1204 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1206 [00:00<?, ? examples/s]

LegalQAEval test examples: 1206


In [7]:
legal_f1_scores = []
n_empty_gold_legal = 0

for ex in legal:
    gold_chars: set = set()
    for a in ex["answers"]:
        gold_chars.update(range(a["start"], a["end"]))
    if not gold_chars:
        n_empty_gold_legal += 1
    _, _, f1 = compute_character_f1(gold_chars, set())
    legal_f1_scores.append(f1)

legal_mean_f1 = statistics.mean(legal_f1_scores) * 100
print(f"Examples with empty gold annotation: {n_empty_gold_legal} / {len(legal)} ({n_empty_gold_legal/len(legal)*100:.1f}%)")
print(f"Empty-prediction char-F1 (LegalQAEval): {legal_mean_f1:.2f}%")

Examples with empty gold annotation: 603 / 1206 (50.0%)
Empty-prediction char-F1 (LegalQAEval): 50.00%


## Summary

In [8]:
print("=" * 50)
print("Empty-prediction baseline (always predict ∅)")
print("=" * 50)
print(f"  ToxicSpans   char-F1: {toxic_mean_f1:.2f}%")
print(f"  LegalQAEval  char-F1: {legal_mean_f1:.2f}%")
print()
print("The score equals the fraction of examples where gold is also empty,")
print("since those are the only cases where empty pred gets F1 = 1.0.")
print(f"  ToxicSpans  empty-gold rate: {n_empty_gold/len(toxic)*100:.1f}%")
print(f"  LegalQAEval empty-gold rate: {n_empty_gold_legal/len(legal)*100:.1f}%")

Empty-prediction baseline (always predict ∅)
  ToxicSpans   char-F1: 48.60%
  LegalQAEval  char-F1: 50.00%

The score equals the fraction of examples where gold is also empty,
since those are the only cases where empty pred gets F1 = 1.0.
  ToxicSpans  empty-gold rate: 48.6%
  LegalQAEval empty-gold rate: 50.0%
